# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [1]:
"""The following generates the Quiz all our models will be evaluated on."""
import itertools
import string
import numpy as np

from dotenv import load_dotenv

load_dotenv("keys.env", verbose=True)

from smolbench.induction.chromatic import (
	ChromaticIntervalsConfig,
	Prompter,
	get_random_exclusive_quiz,
	anneal_intervals
)
from smolbench.evals import (
	Marks
)
from smolbench.evals.openrouter import (
	evaluate
)

template = string.Template(
	"You are a Boolean classifier.\n"
	"\n"
	"Task: determine whether the statement in the Question is logically "
	"possible given the Context.\n"
	"\n"
	"Output format:\n"
	"Return exactly one of these two strings and nothing else:\n"
	"True\n"
	"False\n"
	"\n"
	"Do not output any explanation, punctuation, quotes, labels, code fences, "
	"or extra whitespace."
	"Stop immediately after writing True or False."
	"\n"
	"Context:\n"
	"There is a ceremonial role called the $role, whose job it is to"
	" head the $parade parade. No one else besides the $role is able to head"
	" the $parade parade. The following lists the people who were $role and"
	" the years they were $role:\n"
	"$positive_info\n"
	"\n"
	"Question:\n"
	"Between the years $start and $end, exclusive of the end, could $color"
	" have headed the $parade parade every year?"
)

def query_gen(
	labels_to_intervals: Dict[Color, Intervals],
	interval_to_label: Dict[Intervals, Color],
	seed: int,
) -> Dict[str, str]:
	"""Generates a series of queries"""
	rng: np.random.Generator = np.random.default_rng(seed)
	# Finds max interval.
	n: int = max(interval[1] for interval in interval_to_label)
	for color, intervals in labels_to_intervals.items():
		# Generates a series of true items.
		for start, end in intervals:
			start, end = np.sort(
				rng.choice(range(start, end + 1), size=2, replace=False)
			)
			yield {"color": color, "start": start, "end": end}, True
		# Generates a false proposition.
		invalid_range: Intervals = anneal_intervals(
			itertools.chain(
				(set(interval_to_label.keys()) - set(itertools.chain(*intervals)))
			)
		)
		for start, end in invalid_range:
			start = rng.choice(range(start, end))
			# Binom with p = intervals / n capped at end for a similar-ish
			# distr. to positive accounts.
			end = min(
				end,
				start
				+ rng.binomial(
					end - start + 1,
					np.mean([len(interval) for interval in interval_to_label]) / n,
				)
				+ 1,
			)
			yield {"color": color, "start": start, "end": end}, False


intens_quiz, extens_quiz = get_random_exclusive_quiz(
	ChromaticIntervalsConfig(
            n=250,
            intervals=250 // 4,
            colors=45,
            seed=1776,
        ),
	Prompter(
		template,
		{
			"role": "Twislax",
			"parade": "Gildane",
		},
		query_gen,
	),
)

# Decoder-Only Model
This section tests classical decoder-only models.

In [2]:
decode_intens_eval: Marks = evaluate(intens_quiz, "mistralai/mistral-7b-instruct-v0.1")
decode_extens_eval: Marks = evaluate(extens_quiz, "mistralai/mistral-7b-instruct-v0.1")

0 0 0
0 1 0
' False. Sg was Twislax on years 24 to 34 and 35 to 36. Therefore, Sg could not have headed the Gildane parade between years 163 and 166.' is not a bool.
0 1 1
0 2 1
0 3 1
1 3 1
1 4 1
2 4 1
3 4 1
3 5 1
' False. PD was Twislax only in the year 63.' is not a bool.
3 5 2
3 6 2
4 6 2
4 7 2
5 7 2
5 8 2
5 9 2
' False. Gt was Twislax only in the years 88 to 90.' is not a bool.
5 9 3
5 10 3
5 11 3
6 11 3
6 12 3
6 13 3
6 14 3
7 14 3
7 15 3
7 16 3
7 17 3
8 17 3
9 17 3
9 18 3
10 18 3
10 19 3
10 20 3
11 20 3
12 20 3
13 20 3
14 20 3
15 20 3
15 21 3
16 21 3
16 22 3
16 23 3
16 24 3
16 25 3
16 26 3
17 26 3
17 27 3
18 27 3
18 28 3
' False. Vp was Twislax only in the year 121.' is not a bool.
18 28 4
18 29 4
18 30 4
19 30 4
19 31 4
19 32 4
20 32 4
20 33 4
21 33 4
21 34 4
22 34 4
' False.

The context states that wJ was not Twislax during the years 40 to 42.' is not a bool.
22 34 5
22 35 5
23 35 5
' False.

The context does not list zI as Twislax during the years 110 to 112.' is not a bool.
2

In [4]:
print(decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid)
print(decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid)

52 43 12
38 58 11
